# 03. pandas 핵심 — 표를 다루는 최소한의 문법

## 학습 목표
1. Series / DataFrame 만들고 훑어보기 (`info`, `describe`, `head`)
2. 선택: `[]` vs `.loc` vs `.iloc` — 셋의 차이를 정확히
3. 결측치: `isna`, `fillna`, `dropna`
4. 정렬·`groupby`·`agg`
5. 합치기: `concat` / `merge` — **`DataFrame.append` 는 2.0 에서 삭제됐다**
6. 루프로 행을 붙이면 왜 느린가 (O(n²))
7. `inplace=True` 와 체인 할당(SettingWithCopy) 이야기

> 5·6번은 이 저장소의 옛 노트북(`legacy/Untitled.ipynb`)이 실제로 쓴 방식이고,
> 지금 실행하면 **AttributeError** 로 멈춘다.

In [1]:
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.2


## 1. 만들고 훑어보기

In [2]:
df = pd.DataFrame(
    {
        "이름": ["가영", "나윤", "다은", "라온", "마루"],
        "부서": ["개발", "개발", "디자인", "영업", "디자인"],
        "연차": [3, 7, 1, 5, np.nan],
        "연봉": [4200, 6800, 3100, 5200, 3600],
    }
)
df

,이름,부서,연차,연봉
0,가영,개발,3.0,4200
1,나윤,개발,7.0,6800
2,다은,디자인,1.0,3100
3,라온,영업,5.0,5200
4,마루,디자인,NaN,3600


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   이름      5 non-null      str    
 1   부서      5 non-null      str    
 2   연차      4 non-null      float64
 3   연봉      5 non-null      int64  
dtypes: float64(1), int64(1), str(2)
memory usage: 292.0 bytes


In [4]:
df.describe()          # 숫자 컬럼만. 전체를 보려면 include="all"

,연차,연봉
count,4.000000,5.000000
mean,4.000000,4580.000000
std,2.581989,1466.969666
min,1.000000,3100.000000
25%,2.500000,3600.000000
50%,4.000000,4200.000000
75%,5.500000,5200.000000
max,7.000000,6800.000000


`info()` 에서 확인할 것 두 가지: **Non-Null Count**(결측 개수)와 **Dtype**.
숫자여야 할 컬럼이 `object` 로 잡혀 있으면 그 안에 문자열이 섞여 있다는 뜻이다 (04에서 실제로 만난다).

## 2. 선택 — `[]` vs `.loc` vs `.iloc`

| 문법 | 의미 |
|---|---|
| `df["연봉"]` | **열** 하나 (Series) |
| `df[["이름","연봉"]]` | 여러 **열** |
| `df[0:2]` | **행** 슬라이스 (헷갈림의 주범) |
| `df.loc[행라벨, 열라벨]` | 라벨 기준 — **끝 포함** |
| `df.iloc[행번호, 열번호]` | 위치 기준 — 끝 미포함 |

In [5]:
print(df["연봉"].tolist())
print(df.loc[1:3, ["이름", "연봉"]].to_dict("records"))   # 라벨 1,2,3 → 3개 (끝 포함!)
print(df.iloc[1:3, [0, 3]].to_dict("records"))            # 위치 1,2   → 2개 (끝 제외)

[4200, 6800, 3100, 5200, 3600]
[{'이름': '나윤', '연봉': 6800}, {'이름': '다은', '연봉': 3100}, {'이름': '라온', '연봉': 5200}]
[{'이름': '나윤', '연봉': 6800}, {'이름': '다은', '연봉': 3100}]


같은 `1:3` 인데 결과 개수가 다르다. **`.loc` 은 끝을 포함**한다 — 파이썬 슬라이스와 다른 유일한 지점이라
실무에서 off-by-one 을 자주 만든다.

In [6]:
# 불리언 인덱싱 (NumPy 마스킹과 동일한 원리)
df[(df["연차"] > 2) & (df["부서"] == "개발")]

,이름,부서,연차,연봉
0,가영,개발,3.0,4200
1,나윤,개발,7.0,6800


In [7]:
# query 는 같은 일을 문자열로 — 컬럼명이 길 때 읽기 좋다
df.query("연차 > 2 and 부서 == '개발'")

,이름,부서,연차,연봉
0,가영,개발,3.0,4200
1,나윤,개발,7.0,6800


## 3. 결측치

In [8]:
print(df.isna().sum())                       # 컬럼별 결측 개수
print("\n연차 평균으로 채우기:")
print(df["연차"].fillna(df["연차"].mean()).tolist())
print("\n결측 행 버리기:", len(df.dropna()), "행 남음")

이름    0
부서    0
연차    1
연봉    0
dtype: int64

연차 평균으로 채우기:
[3.0, 7.0, 1.0, 5.0, 4.0]

결측 행 버리기: 4 행 남음


결측치는 **왜 비었는지**에 따라 처리가 달라진다.
* 측정 실패 → 평균/중앙값 대치
* 해당 없음(예: 무직자의 연봉) → 0 이 아니라 `NaN` 유지가 옳을 수 있다
* 수집 누락이 특정 그룹에 몰려 있으면 → 그 자체가 분석 대상이다

아무 생각 없이 `fillna(0)` 하면 평균이 조용히 내려간다.

## 4. 정렬 · groupby · agg

In [9]:
df.sort_values("연봉", ascending=False)

,이름,부서,연차,연봉
1,나윤,개발,7.0,6800
3,라온,영업,5.0,5200
0,가영,개발,3.0,4200
4,마루,디자인,NaN,3600
2,다은,디자인,1.0,3100


In [10]:
df.groupby("부서").agg(
    인원=("이름", "count"),
    평균연봉=("연봉", "mean"),
    최고연봉=("연봉", "max"),
    평균연차=("연차", "mean"),
).round(1)

,인원,평균연봉,최고연봉,평균연차
부서,,,,
개발,2,5500.0,6800,5.0
디자인,2,3350.0,3600,1.0
영업,1,5200.0,5200,5.0


`agg(새이름=("컬럼","함수"))` 형태(named aggregation)를 쓰면 결과 컬럼명이 깔끔하다.
예전 스타일인 `groupby(...).agg({"연봉": ["mean","max"]})` 는 다중 인덱스 컬럼이 생겨 다루기 번거롭다.

## 5. 합치기 — `append` 는 사라졌다

`DataFrame.append` 는 pandas 1.4 에서 deprecate, **2.0 에서 삭제**됐다.

In [11]:
print("DataFrame.append 존재?", hasattr(pd.DataFrame, "append"))

새_직원 = pd.DataFrame([{"이름": "바다", "부서": "영업", "연차": 2, "연봉": 3900}])

try:
    df.append(새_직원)          # 옛 코드가 쓰던 방식
except AttributeError as exc:
    print("AttributeError:", exc)

DataFrame.append 존재? False
AttributeError: 'DataFrame' object has no attribute 'append'


In [12]:
# 지금의 정답
pd.concat([df, 새_직원], ignore_index=True).tail(3)

,이름,부서,연차,연봉
3,라온,영업,5.0,5200
4,마루,디자인,NaN,3600
5,바다,영업,2.0,3900


## 6. 루프 안에서 행 붙이기가 느린 이유

옛 노트북은 이렇게 썼다.
```python
for temp in old_data["info"]:
    new_data = new_data.append(pd.Series(dataSplit(temp), index=new_data.columns), ignore_index=True)
```
`append`(그리고 대체재인 `concat`)는 **매번 전체 데이터를 새로 복사**한다. n 번 반복하면 O(n²)다.
정답은 **파이썬 리스트에 모았다가 마지막에 한 번** DataFrame 으로 만드는 것이다.

In [13]:
rows = [{"a": i, "b": i * 2} for i in range(2000)]

In [14]:
%%timeit -n 1 -r 2
acc = pd.DataFrame([rows[0]])
for row in rows[1:800]:                      # 800개만 해도 이 정도다
    acc = pd.concat([acc, pd.DataFrame([row])], ignore_index=True)

619 ms ± 1.19 ms per loop (mean ± std. dev. of 2 runs, 1 loop each)


In [15]:
%%timeit -n 1 -r 2
pd.DataFrame(rows)                            # 2000개 전체를 한 번에

2.41 ms ± 493 μs per loop (mean ± std. dev. of 2 runs, 1 loop each)


수백 배 차이가 난다. **"리스트에 모아서 마지막에 한 번"** 이 pandas 의 기본 반사여야 한다.

## 7. merge — 키로 붙이기

In [16]:
부서정보 = pd.DataFrame({"부서": ["개발", "디자인", "영업"], "층": [7, 5, 3]})
merged = df.merge(부서정보, on="부서", how="left")
merged

,이름,부서,연차,연봉,층
0,가영,개발,3.0,4200,7
1,나윤,개발,7.0,6800,7
2,다은,디자인,1.0,3100,5
3,라온,영업,5.0,5200,3
4,마루,디자인,NaN,3600,5


`how` 는 `left`(왼쪽 유지) / `inner`(교집합) / `outer`(합집합) / `right`.
**행 수가 변했는지 반드시 확인**한다 — 키가 중복이면 조용히 행이 불어난다.

In [17]:
print(f"merge 전 {len(df)}행 → 후 {len(merged)}행")
print("키 중복 여부:", 부서정보["부서"].duplicated().any())

merge 전 5행 → 후 5행
키 중복 여부: False


## 8. `inplace=True` 와 체인 할당

`inplace=True` 는 메모리를 아껴 주지 않는다(대부분 내부에서 복사한다). 대신 메서드 체이닝을 끊고,
반환값이 `None` 이라 실수를 부른다. **권장하지 않는다.**

In [18]:
result = df.sort_values("연봉", inplace=True)   # 반환값이 없다
print("inplace 의 반환값:", result)

inplace 의 반환값: None


In [19]:
# 권장: 새 변수에 담기 (체이닝 가능)
top = (
    df.dropna(subset=["연차"])
    .assign(연봉_만원=lambda d: d["연봉"])
    .sort_values("연봉_만원", ascending=False)
    .head(3)
)
top

,이름,부서,연차,연봉,연봉_만원
1,나윤,개발,7.0,6800,6800
3,라온,영업,5.0,5200,5200
0,가영,개발,3.0,4200,4200


슬라이스에 값을 대입하는 **체인 할당**(`df[df.a > 0]["b"] = 1`)은 원본에 반영될 수도, 안 될 수도 있다.
pandas 3.0 의 Copy-on-Write 에서는 **반영되지 않는다**. 항상 `.loc` 한 번으로 지정한다.

In [20]:
sample = df.copy()
sample.loc[sample["부서"] == "개발", "연봉"] = 9999   # 이렇게
sample[["이름", "부서", "연봉"]]

,이름,부서,연봉
2,다은,디자인,3100
4,마루,디자인,3600
0,가영,개발,9999
3,라온,영업,5200
1,나윤,개발,9999


## 정리

| 하지 말 것 | 대신 |
|---|---|
| `df.append(...)` | `pd.concat([...])` (2.0 에서 삭제됨) |
| 루프 안에서 concat | 리스트에 모아 마지막에 `pd.DataFrame(rows)` |
| `df[cond]["col"] = v` | `df.loc[cond, "col"] = v` |
| `inplace=True` | 새 변수에 대입 + 체이닝 |
| `fillna(0)` 습관 | 결측의 의미를 먼저 판단 |

다음: **04. 지저분한 실데이터 정제** — 위 문법을 실제 매물 데이터에 적용한다.